# s5 — PCN from scratch: the geometry baseline + the color arms

**This notebook is the first TRAINING stage.** It consumes the s4 pairs from `Ortak/data_s4`
(complete colored GT + PCN-style `meshray_nn` partials) and answers *"is the pipeline working,
and what does color change?"* via three arms trained on identical data/split/seeds:

| arm | input → output | what it isolates |
|---|---|---|
| **A_geo** | `xyz → xyz` | the **s5 baseline** every arm is judged against |
| **B_joint** | `xyzrgb → xyzrgb` | joint geometry+color completion (the proposal's novelty) |
| **C_colin** | `xyzrgb → xyz` | does color **input** help geometry? (A→C = input effect; C→B = joint-output effect) |

The model is **PCN written from scratch in PyTorch** (encoder → coarse decoder → folding decoder),
config-driven in its channel counts, with a pure-torch Chamfer (no CUDA extensions). Sizing:
input 2048 → coarse 1024 → (2×4 folding grid) → fine **8192 = GT size**.

**Run order:** T0 → T7 top-to-bottom (setup, minutes), then the **GATES**: T8 overfit-1, T8b overfit-10 —
*never proceed past a failed gate* — then T9 (the three full runs) and T10 (the A/B/C comparison).
Everything an advisor needs lands on Drive under `Ortak/runs_s5/` (metrics, checkpoints, PNG/HTML panels).

Decisions + rationale: `docs/adr/0006-*`. Terms: `CONTEXT.md`. Honest scope note: with 60 models these
numbers validate the pipeline and measure the A/B/C **delta** — they are not literature-comparable.

In [ ]:
# ============================================================
# Cell T0 — Configuration  (RUN ME FIRST)
# ------------------------------------------------------------
# Everything tunable lives here. The three ARM configs share ONE model/loss codebase;
# an arm changes only channel counts + the color-loss weight. The comparison is valid
# because split, seeds, sizes, and schedule are identical across arms.
# ============================================================
SHARED_ROOT = "/content/drive/MyDrive/Point Cloud Completion"
ORTAK       = f"{SHARED_ROOT}/Ortak"
DATA_S4     = f"{ORTAK}/data_s4"          # s4 handoff (complete/ + partial/<method>/ + manifest.csv)
METHOD      = "meshray_nn"                # the reference partial method (ADR-0005); others = later ablation
RUN_TAG     = "s5"
RUNS_DIR    = f"{ORTAK}/runs_{RUN_TAG}"   # run outputs land here (metrics, checkpoints, viz)

WORK_DIR       = "/content/work_s5"       # local scratch (Drive FUSE is slow on many small files)
LOCAL_COMPLETE = f"{WORK_DIR}/complete"
LOCAL_PARTIAL  = f"{WORK_DIR}/partial"    # meshray_nn only

CATEGORIES     = {"airplane": "02691156", "car": "02958343", "chair": "03001627"}
NAME_BY_SYNSET = {v: k for k, v in CATEGORIES.items()}

# --- sizes (invariant: N_FINE = N_COARSE * GRID_H * GRID_W = the GT size) ---
N_INPUT  = 2048          # points fed to the encoder (partials are ~10-25k -> subsampled)
N_COARSE = 1024          # PCN-canonical coarse size
GRID_H, GRID_W = 2, 4    # folding grid per coarse point (2x4 -> x8)
N_FINE   = N_COARSE * GRID_H * GRID_W    # 8192 == the s4 GT size
N_GT     = 8192          # what every data_s4/complete PLY must contain (asserted in T2)
GRID_SCALE = 0.05        # half-extent of each folded patch (PCN's value)

# --- split & seeds (identical for every arm -> controlled comparison) ---
SPLIT_SEED    = 42       # fixes WHICH models are val
VAL_FRAC      = 0.2      # 16 train / 4 val models per category (on the color-ok subset)
TRAIN_SEED    = 1234     # torch/numpy seed for weight init + train-time subsampling
COLOR_STD_MIN = 0.01     # GT color std below this = colorless model -> excluded from ALL arms

# --- training knobs ---
BATCH_SIZE  = 16
LR          = 1e-4
LR_DECAY    = 0.7        # multiply LR by this ...
LR_STEP     = 50         # ... every LR_STEP epochs
EPOCHS_FULL = 300
ALPHA_SCHED = [(0, 0.01), (50, 0.1), (100, 0.5), (150, 1.0)]  # PCN's fine-loss weight anneal
VIZ_EVERY   = 25         # epochs between val panel renders
CKPT_EVERY  = 10         # epochs between checkpoint/metrics writes (and val evals)
FSCORE_TAU  = 0.01       # F-score threshold (~1% of the near-unit model_normalized frame)

# --- the three arms (in_ch/out_ch slice xyzrgb; color_loss_w only bites when out_ch=6) ---
ARMS = {
    "A_geo":   dict(in_ch=3, out_ch=3, color_loss_w=0.0),  # xyz    -> xyz     (the s5 baseline)
    "B_joint": dict(in_ch=6, out_ch=6, color_loss_w=1.0),  # xyzrgb -> xyzrgb  (joint color completion)
    "C_colin": dict(in_ch=6, out_ch=3, color_loss_w=0.0),  # xyzrgb -> xyz     (color input, geometry out)
}
assert N_FINE == N_GT, "fine output must match the GT size (see ADR-0006 sizing decision)"
print(f"{RUN_TAG}: {METHOD} pairs | input {N_INPUT} -> coarse {N_COARSE} -> fine {N_FINE} (GT {N_GT})")
print("arms:", {k: f"{v['in_ch']}ch -> {v['out_ch']}ch" for k, v in ARMS.items()})

In [ ]:
# ============================================================
# Cell T1 — Mount Drive, install deps, copy the s4 pairs to LOCAL disk
# ------------------------------------------------------------
# Drive FUSE is slow on many small files, so training reads from /content. This copy
# is the only bulk Drive READ; only run outputs (T7+) write back to Drive. ~0.5-1 GB.
# Re-running is a no-op if the local copy already exists (delete WORK_DIR to force).
# ============================================================
import os, shutil, subprocess, time
from google.colab import drive

drive.mount("/content/drive")
assert os.path.isdir(ORTAK), (
    f"\n[STOP] Can't see the shared folder:\n  {ORTAK}\n"
    "Add the 'Point Cloud Completion' shortcut to your My Drive:\n"
    "Drive (web) -> Shared with me -> right-click -> Organize -> Add shortcut to Drive -> My Drive."
)
assert os.path.isdir(f"{DATA_S4}/complete") and os.path.isdir(f"{DATA_S4}/partial/{METHOD}"), (
    f"\n[STOP] {DATA_S4} is missing complete/ or partial/{METHOD}/ — re-run the s4 handoff (Cell Q6)."
)

subprocess.run("pip install -q open3d kaleido", shell=True, check=True)  # PLY IO + PNG export for plotly

import torch
print("torch", torch.__version__, "| GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — Runtime > Change runtime type > GPU")

os.makedirs(WORK_DIR, exist_ok=True)
t0 = time.time()
if not os.path.isdir(LOCAL_COMPLETE):
    shutil.copytree(f"{DATA_S4}/complete", LOCAL_COMPLETE)
if not os.path.isdir(LOCAL_PARTIAL):
    shutil.copytree(f"{DATA_S4}/partial/{METHOD}", LOCAL_PARTIAL)
for f in ("manifest.csv", "viewpoints.csv"):
    shutil.copy(f"{DATA_S4}/{f}", f"{WORK_DIR}/{f}")

n_gt  = sum(len(fs) for _, _, fs in os.walk(LOCAL_COMPLETE))
n_par = sum(len(fs) for _, _, fs in os.walk(LOCAL_PARTIAL))
print(f"local copy ready in {time.time()-t0:.0f}s: {n_gt} complete PLYs, {n_par} {METHOD} partial PLYs")

In [ ]:
# ============================================================
# Cell T2 — Integrity check: disk vs manifest, GT sizes, the color-ok subset
# ------------------------------------------------------------
# Fail LOUDLY before training on bad data. We (1) reconcile files against manifest.csv,
# (2) assert every GT is exactly N_GT xyzrgb points, (3) recompute per-model GT color_std
# and EXCLUDE colorless (textureless) models from ALL arms — every arm must see the SAME
# models, or the A/B/C comparison is confounded by data instead of channels.
# ============================================================
import glob, os
import numpy as np, pandas as pd
import open3d as o3d

def load_ply_xyzrgb(ply_path):
    """(N,6) float32 xyz+rgb, rgb in [0,1] (same helper as s4)."""
    pc = o3d.io.read_point_cloud(ply_path)
    xyz = np.asarray(pc.points); rgb = np.asarray(pc.colors)
    if rgb.shape[0] != xyz.shape[0]:
        rgb = np.zeros_like(xyz)
    return np.concatenate([xyz, rgb], axis=1).astype(np.float32)

man = pd.read_csv(f"{WORK_DIR}/manifest.csv")
man = man[man["method"] == METHOD].reset_index(drop=True)
print(f"manifest: {len(man)} {METHOD} rows, {man['model_id'].nunique()} models")

models = []   # one dict per model: synset, model_id, category, gt (np), partial_paths, color_std
for (synset, mid), grp in man.groupby(["synset", "model_id"], sort=True):
    synset = str(synset).zfill(8)                 # csv parses synsets like 02691156 as ints
    gt_path = f"{LOCAL_COMPLETE}/{synset}/{mid}.ply"
    part_paths = sorted(glob.glob(f"{LOCAL_PARTIAL}/{synset}/{mid}/*.ply"))
    assert os.path.exists(gt_path), f"missing GT: {gt_path}"
    assert len(part_paths) == 8, f"{synset}/{mid}: expected 8 partials, found {len(part_paths)}"
    gt = load_ply_xyzrgb(gt_path)
    assert gt.shape == (N_GT, 6), f"{synset}/{mid}: GT shape {gt.shape} != ({N_GT}, 6)"
    models.append(dict(synset=synset, model_id=mid, category=NAME_BY_SYNSET[synset],
                       gt=gt, partial_paths=part_paths, color_std=float(gt[:, 3:6].std())))

MODELS  = [m for m in models if m["color_std"] > COLOR_STD_MIN]   # every arm uses exactly this subset
dropped = [m for m in models if m["color_std"] <= COLOR_STD_MIN]
print(f"models on disk: {len(models)} | color-ok kept: {len(MODELS)} | dropped as colorless:")
for m in dropped:
    print(f"   {m['category']}/{m['model_id']}  color_std={m['color_std']:.4f}")
print("color-ok per category:", pd.Series([m["category"] for m in MODELS]).value_counts().to_dict())

In [ ]:
# ============================================================
# Cell T3 — Model-level split + the PyTorch Dataset
# ------------------------------------------------------------
# Split BY MODEL, never by pair: 8 views of one model are near-duplicates, so a
# pair-level split would leak shapes into val and make val CD meaninglessly good.
# Per category: shuffle color-ok models with SPLIT_SEED, hold out VAL_FRAC as val.
# The dataset always yields 6 channels (xyzrgb); each ARM slices what it consumes,
# so all arms read literally identical batches. The coarse target = FPS(GT, 1024)
# is precomputed ONCE per model (PCN supervises coarse against a matching-size GT).
# ============================================================
import time
import numpy as np, torch
from torch.utils.data import Dataset, DataLoader

def fps_numpy(points, n, seed=0):
    """Plain geometric FPS on xyz; the color columns ride along. O(n*N) — fast at 8k pts."""
    xyz = points[:, :3]; N = xyz.shape[0]
    if n >= N:
        return points.copy()
    rng = np.random.default_rng(seed)
    sel = np.empty(n, dtype=np.int64); sel[0] = rng.integers(N)
    dist = np.full(N, np.inf)
    for i in range(1, n):
        dist = np.minimum(dist, ((xyz - xyz[sel[i - 1]]) ** 2).sum(1))
        sel[i] = int(np.argmax(dist))
    return points[sel]

t0 = time.time()
for m in MODELS:
    m["gt_coarse"] = fps_numpy(m["gt"], N_COARSE, seed=SPLIT_SEED)
print(f"precomputed {len(MODELS)} coarse GTs ({N_COARSE} pts) in {time.time()-t0:.0f}s")

rng_split = np.random.default_rng(SPLIT_SEED)
train_models, val_models = [], []
for cat in CATEGORIES:
    cat_models = sorted([m for m in MODELS if m["category"] == cat], key=lambda m: m["model_id"])
    order = rng_split.permutation(len(cat_models))
    n_val = max(1, round(VAL_FRAC * len(cat_models)))
    val_models   += [cat_models[i] for i in order[:n_val]]
    train_models += [cat_models[i] for i in order[n_val:]]
print(f"split: {len(train_models)} train / {len(val_models)} val models "
      f"({len(train_models)*8} / {len(val_models)*8} pairs) — identical for every arm")

class PairDataset(Dataset):
    """Yields (partial (N_INPUT,6), gt (N_GT,6), gt_coarse (N_COARSE,6)), float32 xyzrgb.
    train=True  -> a fresh random N_INPUT subset of the partial each time (cheap augmentation)
    train=False -> a per-item FIXED subset (deterministic val / memorizable overfit target)"""
    def __init__(self, model_list, train, views=tuple(range(8))):
        self.items = [(m, v) for m in model_list for v in views]
        self.train = train
        self.partials = [load_ply_xyzrgb(m["partial_paths"][v]) for m, v in self.items]
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        m, v = self.items[i]
        p = self.partials[i]
        rng = np.random.default_rng(None if self.train else 10_000 + i)
        idx = rng.choice(len(p), N_INPUT, replace=len(p) < N_INPUT)
        return p[idx], m["gt"], m["gt_coarse"]

print("PairDataset ready (always 6-channel; arms slice channels at the model boundary).")

In [ ]:
# ============================================================
# Cell T4 — The PCN model, from scratch (config-driven channels)
# ------------------------------------------------------------
# PCN = PointNet-style encoder -> (1) coarse decoder: an MLP emitting N_COARSE points
# directly, (2) fine decoder: FoldingNet-style — each coarse point unfolds a small 2D
# grid patch conditioned on the global feature (2x4 here, so 1024 -> 8192 = GT size).
# in_ch=3 feeds xyz, in_ch=6 feeds xyzrgb; out_ch likewise on the prediction side.
# rgb passes through a sigmoid (colors live in [0,1]); xyz stays linear; fine xyz is
# a residual OFFSET around its coarse seed point (the folding).
# Reference: Yuan et al., "PCN: Point Completion Network" (3DV 2018). Structure
# cross-checked against qinglew/PCN-PyTorch, but written and owned line by line here.
# ============================================================
import torch
import torch.nn as nn

class PCN(nn.Module):
    def __init__(self, in_ch=3, out_ch=3, n_coarse=N_COARSE, grid=(GRID_H, GRID_W), grid_scale=GRID_SCALE):
        super().__init__()
        self.in_ch, self.out_ch, self.n_coarse = in_ch, out_ch, n_coarse
        self.n_fine = n_coarse * grid[0] * grid[1]
        # encoder: shared MLP -> maxpool global -> concat back -> shared MLP -> maxpool
        self.enc1 = nn.Sequential(nn.Conv1d(in_ch, 128, 1), nn.ReLU(), nn.Conv1d(128, 256, 1))
        self.enc2 = nn.Sequential(nn.Conv1d(512, 512, 1), nn.ReLU(), nn.Conv1d(512, 1024, 1))
        # coarse decoder: global feature -> N_COARSE x out_ch
        self.coarse_mlp = nn.Sequential(nn.Linear(1024, 1024), nn.ReLU(),
                                        nn.Linear(1024, 1024), nn.ReLU(),
                                        nn.Linear(1024, n_coarse * out_ch))
        # folding decoder: [global 1024 | coarse seed out_ch | grid uv 2] -> out_ch per fine point
        self.fold = nn.Sequential(nn.Conv1d(1024 + out_ch + 2, 512, 1), nn.ReLU(),
                                  nn.Conv1d(512, 512, 1), nn.ReLU(),
                                  nn.Conv1d(512, out_ch, 1))
        gy, gx = torch.meshgrid(torch.linspace(-grid_scale, grid_scale, grid[0]),
                                torch.linspace(-grid_scale, grid_scale, grid[1]), indexing="ij")
        self.register_buffer("grid", torch.stack([gy, gx], 0).reshape(2, -1))   # (2, grid_n)

    def forward(self, x):                                   # x (B, N_INPUT, in_ch)
        B = x.shape[0]
        f = self.enc1(x.transpose(1, 2))                    # (B, 256, N)
        g = f.max(dim=2, keepdim=True).values               # (B, 256, 1) global so far
        f = torch.cat([f, g.expand(-1, -1, f.shape[2])], 1) # (B, 512, N)
        feat = self.enc2(f).max(dim=2).values               # (B, 1024) the global feature

        coarse = self.coarse_mlp(feat).view(B, self.n_coarse, self.out_ch)
        if self.out_ch == 6:
            coarse = torch.cat([coarse[..., :3], torch.sigmoid(coarse[..., 3:])], -1)

        grid_n = self.grid.shape[1]                                                   # e.g. 8
        grid = self.grid.unsqueeze(0).unsqueeze(2).expand(B, -1, self.n_coarse, -1)   # (B,2,Nc,g)
        seed = coarse.transpose(1, 2).unsqueeze(3).expand(-1, -1, -1, grid_n)         # (B,out,Nc,g)
        glob = feat.unsqueeze(2).unsqueeze(3).expand(-1, -1, self.n_coarse, grid_n)   # (B,1024,Nc,g)
        y = torch.cat([glob, seed, grid], 1).reshape(B, -1, self.n_coarse * grid_n)   # (B,1026+out,Nf)
        y = self.fold(y).transpose(1, 2)                                              # (B, Nf, out)
        center = seed.reshape(B, self.out_ch, -1).transpose(1, 2)                     # (B, Nf, out)
        fine_xyz = center[..., :3] + y[..., :3]              # folded offsets around each coarse seed
        if self.out_ch == 6:
            return coarse, torch.cat([fine_xyz, torch.sigmoid(y[..., 3:])], -1)
        return coarse, fine_xyz

_p = PCN(in_ch=6, out_ch=6)
_n = sum(p.numel() for p in _p.parameters())
with torch.no_grad():
    _c, _f = _p(torch.randn(2, N_INPUT, 6))
print(f"PCN ok: {_n/1e6:.1f}M params | coarse {tuple(_c.shape)} fine {tuple(_f.shape)}")
del _p, _c, _f

In [ ]:
# ============================================================
# Cell T5 — Chamfer (pure torch, memory-safe), losses, metrics
# ------------------------------------------------------------
# No CUDA extensions. The trick: the 8192x8192 distance matrix exists only inside
# no_grad to find nearest-neighbour INDICES, then distances are recomputed on the
# gathered pairs — mathematically the same subgradient, but backward never holds a
# 268 MB matrix per sample. Color correspondence reuses the same indices (matching
# is GEOMETRIC — xyz only; a 6D chamfer mixing meters with rgb is a banked ablation).
# Conventions (stated so future numbers are unambiguous), both reported x 1e3:
#   CD-L1 = ( mean_a min_b ||a-b||   + mean_b min_a ||b-a||   ) / 2     [norms]
#   CD-L2 = ( mean_a min_b ||a-b||^2 + mean_b min_a ||b-a||^2 ) / 2     [squared]
# Training uses CD-L1 (PCN-style; less dominated by outliers).
# ============================================================
import numpy as np, torch

def nn_dists(a, b):
    """a (Na,3), b (Nb,3) -> (d_ab, idx_ab, d_ba, idx_ba). Euclidean NORMS, differentiable
    through the gathered pairs; the full distance matrix never enters the autograd graph."""
    with torch.no_grad():
        d = torch.cdist(a, b)
        idx_ab = d.argmin(dim=1); idx_ba = d.argmin(dim=0)
    d_ab = (a - b[idx_ab]).pow(2).sum(-1).clamp_min(1e-12).sqrt()
    d_ba = (b - a[idx_ba]).pow(2).sum(-1).clamp_min(1e-12).sqrt()
    return d_ab, idx_ab, d_ba, idx_ba

def chamfer_batch(pred, gt, need_idx=False):
    """pred (B,N,3), gt (B,M,3) -> {'cd_l1','cd_l2' (batch-mean tensors), 'idx' per sample}."""
    l1 = 0.0; l2 = 0.0; idxs = []
    for i in range(pred.shape[0]):
        d_ab, idx_ab, d_ba, idx_ba = nn_dists(pred[i], gt[i])
        l1 = l1 + (d_ab.mean() + d_ba.mean()) / 2
        l2 = l2 + ((d_ab ** 2).mean() + (d_ba ** 2).mean()) / 2
        if need_idx:
            idxs.append((idx_ab, idx_ba))
    B = pred.shape[0]
    return {"cd_l1": l1 / B, "cd_l2": l2 / B, "idx": idxs}

def color_sqerr_batch(pred, gt, idxs):
    """Symmetric per-point rgb squared error at the xyz-NN correspondence, averaged over
    directions AND channels -> a per-channel MSE. pred/gt are (B,*,6)."""
    tot = 0.0
    for i, (idx_ab, idx_ba) in enumerate(idxs):
        pr, gr = pred[i, :, 3:], gt[i, :, 3:]
        tot = tot + ((pr - gr[idx_ab]) ** 2).mean() + ((gr - pr[idx_ba]) ** 2).mean()
    return tot / (2 * len(idxs))

def fscore_batch(pred, gt, tau=FSCORE_TAU):
    """F-score@tau: harmonic mean of precision/recall at distance tau (PCN benchmark metric)."""
    fs = 0.0
    for i in range(pred.shape[0]):
        d_ab, _, d_ba, _ = nn_dists(pred[i], gt[i])
        P = (d_ab < tau).float().mean(); R = (d_ba < tau).float().mean()
        fs = fs + (2 * P * R / (P + R) if float(P + R) > 0 else torch.zeros((), device=pred.device))
    return fs / pred.shape[0]

def alpha_at(epoch):
    """PCN's anneal of the fine-loss weight (ALPHA_SCHED is sorted)."""
    a = ALPHA_SCHED[0][1]
    for e, v in ALPHA_SCHED:
        if epoch >= e:
            a = v
    return a

def compute_loss(coarse, fine, gt, gt_coarse, epoch, color_w):
    """L = CD-L1(coarse, gt_coarse) + alpha * CD-L1(fine, gt)  [+ color_w * rgb MSE terms].
    Returns (loss, parts) — parts logged separately so color never hides geometry."""
    need = color_w > 0
    cc = chamfer_batch(coarse[..., :3], gt_coarse[..., :3], need_idx=need)
    cf = chamfer_batch(fine[..., :3],   gt[..., :3],        need_idx=need)
    a = alpha_at(epoch)
    loss = cc["cd_l1"] + a * cf["cd_l1"]
    parts = {"cd_coarse": float(cc["cd_l1"]), "cd_fine_l1": float(cf["cd_l1"]),
             "cd_fine_l2": float(cf["cd_l2"]), "color": 0.0}
    if need:
        col = color_sqerr_batch(coarse, gt_coarse, cc["idx"]) + color_sqerr_batch(fine, gt, cf["idx"])
        loss = loss + color_w * col
        parts["color"] = float(col)
    return loss, parts

@torch.no_grad()
def eval_metrics(model, loader, device, color_out):
    """Fine-output metrics: CD-L1/L2 (x1e3), F@tau, and (if color_out) rgb MSE + PSNR at NN."""
    model.eval()
    agg = {"cd_l1": 0.0, "cd_l2": 0.0, "fscore": 0.0, "color_mse": 0.0}; n = 0
    for partial, gt, gtc in loader:
        partial, gt = partial.to(device), gt.to(device)
        coarse, fine = model(partial[..., :model.in_ch])
        cf = chamfer_batch(fine[..., :3], gt[..., :3], need_idx=color_out)
        b = len(partial); n += b
        agg["cd_l1"] += float(cf["cd_l1"]) * b
        agg["cd_l2"] += float(cf["cd_l2"]) * b
        agg["fscore"] += float(fscore_batch(fine[..., :3], gt[..., :3])) * b
        if color_out:
            agg["color_mse"] += float(color_sqerr_batch(fine, gt, cf["idx"])) * b
    out = {k: v / n for k, v in agg.items()}
    out["cd_l1"] *= 1e3; out["cd_l2"] *= 1e3
    out["color_psnr"] = float(-10 * np.log10(out["color_mse"] + 1e-12)) if color_out else float("nan")
    if not color_out:
        out["color_mse"] = float("nan")
    return out

print("losses + metrics ready (CD conventions documented above).")

In [ ]:
# ============================================================
# Cell T6 — Visualization: pair panels + training curves (inline AND saved to Drive)
# ------------------------------------------------------------
# "Really see the results": every render shows inline AND writes into the run's viz/
# folder as interactive HTML + PNG (kaleido), so the advisor can browse results on
# Drive without running anything. Clouds without rgb draw in neutral steel-blue;
# colored clouds use their real rgb.
# ============================================================
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def _marker(rgb, size=1.6):
    if rgb is not None and len(rgb) and float(np.asarray(rgb).std()) > 1e-6:
        col = [f"rgb({int(r*255)},{int(g*255)},{int(b*255)})" for r, g, b in np.clip(rgb, 0, 1)]
    else:
        col = "steelblue"
    return dict(size=size, color=col)

def panel(clouds, title, save_stem=None, max_pts=9000, height=420):
    """clouds: list of (arr (N,3|6), subtitle) -> one row of 3D scatters, shared layout."""
    fig = make_subplots(rows=1, cols=len(clouds), specs=[[{"type": "scene"}] * len(clouds)],
                        subplot_titles=[t for _, t in clouds])
    for i, (arr, _t) in enumerate(clouds, 1):
        arr = np.asarray(arr)
        if len(arr) > max_pts:
            s = np.random.choice(len(arr), max_pts, replace=False); arr = arr[s]
        rgb = arr[:, 3:6] if arr.shape[1] >= 6 else None
        fig.add_trace(go.Scatter3d(x=arr[:, 0], y=arr[:, 1], z=arr[:, 2], mode="markers",
                                   marker=_marker(rgb)), row=1, col=i)
    fig.update_layout(title=title, height=height, margin=dict(l=0, r=0, t=50, b=0), showlegend=False)
    fig.update_scenes(aspectmode="data")
    if save_stem:
        os.makedirs(os.path.dirname(save_stem), exist_ok=True)
        fig.write_html(save_stem + ".html")
        try:
            fig.write_image(save_stem + ".png", width=340 * len(clouds), height=height, scale=2)
        except Exception as e:
            print(f"(png export skipped: {e})")
    fig.show()

def curves(df, save_stem=None):
    """df = the run's metrics.csv (one row per epoch). Train CDs are already x1e3."""
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=["train losses (CD x1e3 / color MSE)", "val CD-L1 x1e3 (fine)"])
    for col in ("cd_coarse", "cd_fine_l1", "color"):
        if col in df:
            fig.add_trace(go.Scatter(x=df["epoch"], y=df[col], name=col), row=1, col=1)
    if "val_cd_l1" in df:
        v = df.dropna(subset=["val_cd_l1"])
        fig.add_trace(go.Scatter(x=v["epoch"], y=v["val_cd_l1"], name="val_cd_l1",
                                 mode="lines+markers"), row=1, col=2)
    fig.update_layout(height=360, margin=dict(l=0, r=0, t=50, b=0))
    if save_stem:
        fig.write_html(save_stem + ".html")
        try:
            fig.write_image(save_stem + ".png", width=1000, height=360, scale=2)
        except Exception:
            pass
    fig.show()

@torch.no_grad()
def render_samples(model, dataset, indices, device, title, save_stem=None):
    """The fixed panel: partial | coarse | fine | GT for each requested dataset index."""
    model.eval()
    for j, i in enumerate(indices):
        partial, gt, gtc = dataset[i]
        x = torch.from_numpy(partial[None]).to(device)
        coarse, fine = model(x[..., :model.in_ch])
        m, v = dataset.items[i]
        panel([(partial, f"partial ({N_INPUT})"),
               (coarse[0].cpu().numpy(), f"coarse ({model.n_coarse})"),
               (fine[0].cpu().numpy(), f"fine ({model.n_fine})"),
               (gt, f"GT ({N_GT})")],
              f"{title} — {m['category']}/{m['model_id']} view {v:02d}",
              save_stem=(f"{save_stem}_{j:02d}" if save_stem else None))

print("viz ready (panels + curves, saved to the run's viz/ folder).")

In [ ]:
# ============================================================
# Cell T7 — The training loop (ONE function for overfit gates AND full runs)
# ------------------------------------------------------------
# One code path for every run keeps the arms comparable by construction: only the
# arm config (channels + color weight) and the dataset differ. Per epoch: train;
# every CKPT_EVERY epochs (and at the end): val metrics, best/latest checkpoints,
# metrics.csv — all in the run's Drive folder. resume=True continues after a Colab
# disconnect (restores model/optimizer/scheduler/epoch from latest.pt); a finished
# run resumes into a no-op and just returns its history.
# ============================================================
import json, os, time
import numpy as np, pandas as pd
import torch
from torch.utils.data import DataLoader

def make_run_dir(run_name):
    d = f"{RUNS_DIR}/{run_name}"
    os.makedirs(f"{d}/checkpoints", exist_ok=True)
    os.makedirs(f"{d}/viz", exist_ok=True)
    return d

def run_training(run_name, arm, train_set, val_set, epochs, batch_size=BATCH_SIZE,
                 viz_every=VIZ_EVERY, ckpt_every=CKPT_EVERY, viz_indices=None, resume=False,
                 lr=LR, lr_step=LR_STEP, lr_decay=LR_DECAY):
    # NOTE: the scheduler steps once per EPOCH. The default (LR_STEP=50) is tuned for the
    # full runs (24 steps/epoch). Overfit gates run 1 step/epoch, so they MUST override
    # lr/lr_step (constant LR) or the LR collapses mid-gate and color never learns —
    # found the hard way on the first Colab run (geometry memorized, color froze at init).
    cfg = ARMS[arm]
    torch.manual_seed(TRAIN_SEED); np.random.seed(TRAIN_SEED)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        print("WARNING: no GPU — this will crawl. Runtime > Change runtime type > GPU.")
    model = PCN(in_ch=cfg["in_ch"], out_ch=cfg["out_ch"]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=lr_step, gamma=lr_decay)
    run_dir = make_run_dir(run_name)
    with open(f"{run_dir}/config.json", "w") as fh:
        json.dump(dict(arm=arm, **cfg, method=METHOD, n_input=N_INPUT, n_coarse=N_COARSE,
                       n_fine=N_FINE, epochs=epochs, batch_size=batch_size, lr=lr,
                       lr_step=lr_step, lr_decay=lr_decay, split_seed=SPLIT_SEED,
                       train_seed=TRAIN_SEED, train_pairs=len(train_set),
                       val_pairs=(len(val_set) if val_set is not None else 0)), fh, indent=2)

    start_epoch, best = 0, float("inf")
    ck_latest = f"{run_dir}/checkpoints/latest.pt"
    mpath = f"{run_dir}/metrics.csv"
    rows = []
    if resume and os.path.exists(ck_latest):
        ck = torch.load(ck_latest, map_location=device)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"])
        start_epoch, best = ck["epoch"] + 1, ck["best"]
        if os.path.exists(mpath):
            rows = [r for r in pd.read_csv(mpath).to_dict("records") if r["epoch"] < start_epoch]
        print(f"resumed {run_name} at epoch {start_epoch} (best val CD-L1 x1e3: {best:.3f})")

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = (DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=2)
                  if val_set is not None else None)
    t0 = time.time()
    for epoch in range(start_epoch, epochs):
        model.train()
        ep = {"cd_coarse": 0.0, "cd_fine_l1": 0.0, "cd_fine_l2": 0.0, "color": 0.0}; n = 0
        for partial, gt, gtc in train_loader:
            partial, gt, gtc = partial.to(device), gt.to(device), gtc.to(device)
            coarse, fine = model(partial[..., :cfg["in_ch"]])
            tgt_gt  = gt  if cfg["out_ch"] == 6 else gt[..., :3]
            tgt_gtc = gtc if cfg["out_ch"] == 6 else gtc[..., :3]
            loss, parts = compute_loss(coarse, fine, tgt_gt, tgt_gtc, epoch, cfg["color_loss_w"])
            opt.zero_grad(); loss.backward(); opt.step()
            b = len(partial); n += b
            for k in ep:
                ep[k] += parts[k] * b
        sched.step()
        row = {"epoch": epoch,
               **{k: (v / n * 1e3 if k.startswith("cd") else v / n) for k, v in ep.items()},
               "alpha": alpha_at(epoch), "lr": sched.get_last_lr()[0],
               "val_cd_l1": float("nan"), "val_cd_l2": float("nan"),
               "val_fscore": float("nan"), "val_color_mse": float("nan")}
        is_ckpt = (epoch % ckpt_every == ckpt_every - 1) or (epoch == epochs - 1)
        if val_loader is not None and is_ckpt:
            vm = eval_metrics(model, val_loader, device, color_out=cfg["out_ch"] == 6)
            row.update(val_cd_l1=vm["cd_l1"], val_cd_l2=vm["cd_l2"],
                       val_fscore=vm["fscore"], val_color_mse=vm["color_mse"])
            if vm["cd_l1"] < best:
                best = vm["cd_l1"]
                torch.save({"model": model.state_dict(), "epoch": epoch, "best": best,
                            "arm": arm, "cfg": cfg}, f"{run_dir}/checkpoints/best.pt")
        rows.append(row)
        if is_ckpt:
            torch.save({"model": model.state_dict(), "opt": opt.state_dict(),
                        "sched": sched.state_dict(), "epoch": epoch, "best": best,
                        "arm": arm, "cfg": cfg}, ck_latest)
            pd.DataFrame(rows).to_csv(mpath, index=False)
            print(f"[{run_name}] ep {epoch:4d} | train cd_fine_l1 x1e3 {row['cd_fine_l1']:8.3f} "
                  f"color {row['color']:.5f} | val cd_l1 {row['val_cd_l1']:8.3f} "
                  f"F@1% {row['val_fscore']:.3f} | {time.time()-t0:.0f}s")
        if viz_indices and ((epoch % viz_every == viz_every - 1) or (epoch == epochs - 1)):
            ds = val_set if val_set is not None else train_set
            render_samples(model, ds, viz_indices, device, f"{run_name} @ epoch {epoch}",
                           save_stem=f"{run_dir}/viz/epoch{epoch:04d}")
    pd.DataFrame(rows).to_csv(mpath, index=False)
    return model, pd.DataFrame(rows), run_dir, best

print("run_training ready.")

---
## The overfit ladder — GATES before any full run

The advisor's sanity check: a pipeline that cannot **memorize** a tiny sample is broken, and no
full-training number from it can be trusted. Fixed input subsets (`train=False` sampling) make the
mapping exactly memorizable. **PASS =** fine **CD-L2 ×1e3 < 0.3** on the memorized pair(s) *and* the
panel is visually indistinguishable from GT (arm B: colors match too, **color MSE < 1e-3**).
**FAIL = stop and debug** — do not run T9.

The gates run **2000 steps at a constant LR (5e-4, no decay)**: the full-run schedule decays per
*epoch*, and an overfit epoch is a single optimizer step — reusing it starves the slow learner.
Geometry memorizes in the first ~100 steps; **color is the slow learner these gates exist to check**
(first Colab run: geometry passed at 0.024, color froze at init under the decayed LR).

In [ ]:
# ============================================================
# Cell T8 — GATE 1: overfit-1 (one pair, arms A and B)
# ------------------------------------------------------------
# Batch of 1, 2000 steps at a CONSTANT overfit LR. Both the geometry wiring (A) and
# the color wiring (B) must memorize. The end-of-run panel goes to runs_s5/overfit1_<arm>/viz/.
# First-Colab-run lesson (2026-07-12): with the full-run LR schedule (decay per epoch
# = per STEP here) and only 400 steps, geometry memorized (cd_l2 x1e3 = 0.024) but
# color froze near init (mse 0.103) — the gate FAILed on optimization, not wiring.
# Hence: more steps + constant, slightly higher LR. Geometry converges early anyway;
# color is the slow learner the gate exists to check.
# ============================================================
OVERFIT1_EPOCHS = 2000    # 1 step/epoch at batch 1 -> 2000 optimizer steps
OVERFIT_LR      = 5e-4    # constant for gates (lr_step below disables decay);
                          # the full runs keep the LR/LR_STEP schedule untouched
GATE_CDL2_MAX   = 0.3     # x1e3, on the memorized pair (provisional; panels are the real judge)
GATE_CMSE_MAX   = 1e-3

m0  = next(m for m in train_models if m["category"] == "airplane")
ds1 = PairDataset([m0], train=False, views=(0,))
print(f"overfit-1 target: {m0['category']}/{m0['model_id']} view 00")

overfit1 = {}
for arm in ("A_geo", "B_joint"):
    model, hist, run_dir, _ = run_training(
        f"overfit1_{arm}", arm, ds1, ds1, OVERFIT1_EPOCHS, batch_size=1,
        viz_every=OVERFIT1_EPOCHS, ckpt_every=250, viz_indices=[0],
        lr=OVERFIT_LR, lr_step=10**9)   # constant LR: no decay within the gate
    last = hist.iloc[-1]
    ok_geo = last["val_cd_l2"] < GATE_CDL2_MAX
    ok_col = (last["val_color_mse"] < GATE_CMSE_MAX) if ARMS[arm]["out_ch"] == 6 else True
    overfit1[arm] = bool(ok_geo and ok_col)
    print(f"GATE overfit-1 [{arm}]: cd_l2 x1e3 = {last['val_cd_l2']:.4f} "
          f"(< {GATE_CDL2_MAX}?) color_mse = {last['val_color_mse']:.5f} "
          f"-> {'PASS' if overfit1[arm] else 'FAIL — STOP AND DEBUG'}")
assert all(overfit1.values()), "overfit-1 failed — fix the model/loss before anything else."

In [ ]:
# ============================================================
# Cell T8b — GATE 2: overfit-10 (ten models, one view each, arms A and B)
# ------------------------------------------------------------
# Proves capacity + batching: 10 distinct shapes memorized THROUGH minibatches.
# Same pass criteria as GATE 1, averaged over the 10 pairs. Same constant-LR
# reasoning as T8 (1 step/epoch at batch 10).
# ============================================================
OVERFIT10_EPOCHS = 2000

by_cat = {c: [m for m in train_models if m["category"] == c] for c in CATEGORIES}
ten  = by_cat["airplane"][:4] + by_cat["car"][:3] + by_cat["chair"][:3]
ds10 = PairDataset(ten, train=False, views=(0,))
print("overfit-10 targets:", [f"{m['category']}/{m['model_id'][:8]}" for m in ten])

overfit10 = {}
for arm in ("A_geo", "B_joint"):
    model, hist, run_dir, _ = run_training(
        f"overfit10_{arm}", arm, ds10, ds10, OVERFIT10_EPOCHS, batch_size=10,
        viz_every=OVERFIT10_EPOCHS, ckpt_every=250, viz_indices=[0, 4, 7],
        lr=OVERFIT_LR, lr_step=10**9)
    last = hist.iloc[-1]
    ok_geo = last["val_cd_l2"] < GATE_CDL2_MAX
    ok_col = (last["val_color_mse"] < GATE_CMSE_MAX) if ARMS[arm]["out_ch"] == 6 else True
    overfit10[arm] = bool(ok_geo and ok_col)
    print(f"GATE overfit-10 [{arm}]: cd_l2 x1e3 = {last['val_cd_l2']:.4f} "
          f"color_mse = {last['val_color_mse']:.5f} "
          f"-> {'PASS' if overfit10[arm] else 'FAIL — STOP AND DEBUG'}")
assert all(overfit10.values()), "overfit-10 failed — debug capacity/batching before full runs."

---
## Full runs + the comparison

Three arms, identical split/seeds/schedule — only channels and the color-loss weight differ.
Expect very roughly **20–40 min per arm** on a T4/L4 (24 steps × 300 epochs + evals). After a Colab
disconnect: re-run T0–T7, set `RESUME = True` below, and re-run — finished arms no-op, the
interrupted one continues from its last checkpoint on Drive.

In [ ]:
# ============================================================
# Cell T9 — FULL RUNS: A_geo, B_joint, C_colin on the 16/4-per-category split
# ------------------------------------------------------------
# The fixed viz panel = 2 val models per category (view 00), the SAME indices at
# every render and for every arm, so panels are comparable across epochs AND arms.
# ============================================================
RESUME = False    # set True to continue interrupted runs after a disconnect

train_set = PairDataset(train_models, train=True)
val_set   = PairDataset(val_models,  train=False)

viz_idx, _seen = [], {c: 0 for c in CATEGORIES}
for i, (m, v) in enumerate(val_set.items):
    if v == 0 and _seen[m["category"]] < 2:
        viz_idx.append(i); _seen[m["category"]] += 1
print("fixed panel samples:",
      [f"{val_set.items[i][0]['category']}/{val_set.items[i][0]['model_id'][:8]}" for i in viz_idx])

results = {}
for arm in ("A_geo", "B_joint", "C_colin"):
    print(f"\n=== {arm}  {ARMS[arm]} ===")
    model, hist, run_dir, best = run_training(arm, arm, train_set, val_set, EPOCHS_FULL,
                                              viz_indices=viz_idx, resume=RESUME)
    results[arm] = dict(hist=hist, run_dir=run_dir, best=best)
    curves(hist, save_stem=f"{run_dir}/viz/curves")
    print(f"{arm}: best val CD-L1 x1e3 = {best:.3f}")

In [ ]:
# ============================================================
# Cell T10 — THE COMPARISON: A vs B vs C per category, on identical val pairs
# ------------------------------------------------------------
# Loads each arm's BEST checkpoint, evaluates the SAME val pairs, prints the table
# (the week's finding), renders all arms side by side on the SAME samples, and
# writes everything to runs_s5/comparison/ on Drive. "MEAN" = the whole val set.
# ============================================================
import os
import pandas as pd
import torch
from torch.utils.data import DataLoader

device  = "cuda" if torch.cuda.is_available() else "cpu"
cmp_dir = f"{RUNS_DIR}/comparison"
os.makedirs(cmp_dir, exist_ok=True)

rows, nets = [], {}
for arm, cfg in ARMS.items():
    ck = torch.load(f"{RUNS_DIR}/{arm}/checkpoints/best.pt", map_location=device)
    net = PCN(in_ch=cfg["in_ch"], out_ch=cfg["out_ch"]).to(device)
    net.load_state_dict(ck["model"]); net.eval()
    nets[arm] = net
    for cat in list(CATEGORIES) + ["MEAN"]:
        ms = val_models if cat == "MEAN" else [m for m in val_models if m["category"] == cat]
        loader = DataLoader(PairDataset(ms, train=False), batch_size=BATCH_SIZE)
        vm = eval_metrics(net, loader, device, color_out=cfg["out_ch"] == 6)
        rows.append(dict(arm=arm, category=cat, best_epoch=int(ck["epoch"]),
                         **{k: round(float(v), 4) for k, v in vm.items()}))
res = pd.DataFrame(rows)
print(res.pivot(index="category", columns="arm",
                values="cd_l1").to_string(), "\n   (val fine CD-L1 x1e3, lower is better)\n")
print(res.to_string(index=False))
res.to_csv(f"{cmp_dir}/summary.csv", index=False)

# side-by-side: partial | A | C | B | GT on the same fixed val samples
full_val = PairDataset(val_models, train=False)
viz_idx, _seen = [], {c: 0 for c in CATEGORIES}
for i, (m, v) in enumerate(full_val.items):
    if v == 0 and _seen[m["category"]] < 2:
        viz_idx.append(i); _seen[m["category"]] += 1
for j, i in enumerate(viz_idx):
    partial, gt, _ = full_val[i]
    m, v = full_val.items[i]
    clouds = [(partial, "partial")]
    for arm in ("A_geo", "C_colin", "B_joint"):
        x = torch.from_numpy(partial[None]).to(device)
        with torch.no_grad():
            _, fine = nets[arm](x[..., :nets[arm].in_ch])
        clouds.append((fine[0].cpu().numpy(), f"{arm} fine"))
    clouds.append((gt, "GT"))
    panel(clouds, f"A/C/B on {m['category']}/{m['model_id']} view {v:02d}",
          save_stem=f"{cmp_dir}/sample_{j:02d}")

with open(f"{cmp_dir}/summary.md", "w") as fh:
    fh.write("# s5 comparison — A_geo vs B_joint vs C_colin\n\n")
    fh.write(f"- data: {len(MODELS)} color-ok models ({len(train_models)} train / "
             f"{len(val_models)} val), {METHOD} partials, GT {N_GT} pts\n")
    fh.write("- CD x1e3 lower=better; F-score@1% higher=better; color metrics apply to B only\n")
    fh.write("- caveat: 60-model dataset — pipeline validation + arm DELTAS, not literature numbers\n\n")
    try:
        fh.write(res.to_markdown(index=False))
    except ImportError:
        fh.write(res.to_string(index=False))
print("comparison written ->", cmp_dir)